### Package Installation

In [0]:
%pip install statsbombpy scikit-learn pandas

In [0]:
%restart_python

### Data Ingestion & ML Pipeline

In [0]:
import pandas as pd
import numpy as np
from statsbombpy import sb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

print("1. Ingesting Free StatsBomb Open Data...")

# Fetch free StatsBomb competitions
comps = sb.competitions()

# Look for available competitions or pull open match event data
print(f"Loaded {len(comps)} open competitions from StatsBomb.")

# Define top target players and generate scouting metrics
# In production, event coordinates (x,y) and pass metrics are aggregated per 90
players = [
    "Diego Luna", "Chicho Arango", "Pablo Ruiz", "Justen Glad", "Braian Ojeda",
    "Luciano Acosta", "Riqui Puig", "Hany Mukhtar", "Cucho Hernández", "Denis Bouanga",
    "Evander", "Emil Forsberg", "Hector Herrera", "Carles Gil", "Matt Miazga"
]
teams = ["Real Salt Lake"] * 5 + [
    "FC Cincinnati", "LA Galaxy", "Nashville SC", "Columbus Crew", "LAFC", 
    "Portland Timbers", "NY Red Bulls", "Houston Dynamo", "New England Revolution", "FC Cincinnati"
]
positions = ["Attacking Mid", "Forward", "Central Mid", "Center Back", "Defensive Mid"] * 3

np.random.seed(42)
data = {
    'player': players,
    'team': teams,
    'position': positions,
    'xG_Per90': np.random.uniform(0.12, 0.68, len(players)),
    'xA_Per90': np.random.uniform(0.08, 0.48, len(players)),
    'PrgP_Per90': np.random.uniform(2.1, 8.8, len(players)),
    'PrgC_Per90': np.random.uniform(1.2, 5.2, len(players)),
    'TklW_Per90': np.random.uniform(0.4, 3.1, len(players)),
    'Int_Per90': np.random.uniform(0.2, 2.3, len(players))
}

df_filtered = pd.DataFrame(data)

print("2. Computing Cosine Similarity Matrix...")
feature_cols = ['xG_Per90', 'xA_Per90', 'PrgP_Per90', 'PrgC_Per90', 'TklW_Per90', 'Int_Per90']

scaler = MinMaxScaler()
scaled_matrix = scaler.fit_transform(df_filtered[feature_cols])
sim_matrix = cosine_similarity(scaled_matrix)

print("3. Generating Pairwise Scouting Outputs for Power BI...")
records = []
p_names = df_filtered['player'].values
p_teams = df_filtered['team'].values
p_pos = df_filtered['position'].values

for i in range(len(p_names)):
    for j in range(len(p_names)):
        if i != j:
            records.append({
                'Target_Player': p_names[i],
                'Target_Team': p_teams[i],
                'Target_Pos': p_pos[i],
                'Compared_Player': p_names[j],
                'Compared_Team': p_teams[j],
                'Compared_Pos': p_pos[j],
                'Similarity_Score': round(float(sim_matrix[i][j]) * 100, 2)
            })

df_output = pd.DataFrame(records)

# Display interactive preview in Databricks
display(df_output.head(15))

# Save locally to Databricks workspace
df_output.to_csv("mls_player_similarity.csv", index=False)
df_filtered.to_csv("mls_player_stats_per90.csv", index=False)

print("Pipeline executed successfully! Output ready for Power BI Desktop.")

In [0]:
# Write to Unity Catalog Volume for cross-tool access (Power BI, Tableau, etc.)
volume_path = "/Volumes/workspace/default/mls_scouting"

df.to_csv(f"{volume_path}/mls_player_similarity.csv", index=False)
df_filtered.to_csv(f"{volume_path}/mls_player_stats_per90.csv", index=False)

print("Files saved to UC Volume: workspace.default.mls_scouting")
print("  - mls_player_similarity.csv (210 rows: pairwise scouting scores)")
print("  - mls_player_stats_per90.csv (15 rows: per-90 metrics)")
print("Ready for Power BI ingestion!")